# Dense captioning generation

1. Get image url from roboflow
2. Query openrouter with image and get dense captinos
3. store url and dense captions in json format

In [2]:
import roboflow_client as rc
import dotenv
import os
import json
import time
import tqdm
import base64
from io import BytesIO
from PIL import Image
import requests
from tqdm.notebook import tqdm as tqdmnote
import openai as opi

for key in os.environ.keys():
    os.environ.pop(key)

dotenv.load_dotenv()

True

In [4]:
rf_client = rc.RoboflowAPIClient(os.getenv("ROBOFLOW_API_KEY"), os.getenv("ROBOFLOW_WORKSPACE_ID"), os.getenv("ROBOFLOW_PROJECT_ID"))
print(os.getenv("ROBOFLOW_PROJECT_ID"))

# Uncomment the following to generate metadata for a given dataset

with open("./img-metadata.json", "w", encoding="utf-8") as file:
    project_metadata = {
        "worskpace_id": os.getenv("ROBOFLOW_WORKSPACE_ID"),
        "project_id": os.getenv("ROBOFLOW_PROJECT_ID")
    }
    img_data = {}
    for i, img in enumerate(rf_client.yield_images(), 1):
        id = img["id"]
        orig_url = rf_client.get_image_metadata(id)["urls"]["original"]
        annotation = rf_client.get_image_metadata(id)["annotation"]["textAnnotations"][0]["prefix"]
        # fix for not getting hazard images
        # if(annotation != "01"): continue
        img_data[id] = {"url": orig_url, "label": annotation}
        print(f"Recieved image no. {i} with id {id}")
    json.dump({"metadata": project_metadata, "image_data": img_data}, file, indent=4)
        
    

vlm-in-context-anom-haz-annot-1p23m
Recieved image no. 1 with id r3Bbl9RVhSDU8ZVPf7uK
Recieved image no. 2 with id VgUi4JygVBe9D9c9oA97
Recieved image no. 3 with id 781rGblmCTHhWj3obtbF
Recieved image no. 4 with id SifG7P9TyvQP4dbMsL6C
Recieved image no. 5 with id Pbt9qvzC3Bid8UNNK82L
Recieved image no. 6 with id R2LYctMICBL2nOZCgzTk
Recieved image no. 7 with id Q8lhcERaysiLopxEB4DG
Recieved image no. 8 with id UGMZ3NrhXMmy6PQSLTpF
Recieved image no. 9 with id sFJx21em7clAORpQbZKj
Recieved image no. 10 with id xkVpJNGnOSGgu1jN9izM
Recieved image no. 11 with id 8kpjVaevGcvKrjHxaKcf
Recieved image no. 12 with id 1TrnNJfWdQvhFuPYjH45
Recieved image no. 13 with id PsTA2bbgWfwixYIXdpZM
Recieved image no. 14 with id UF0uX4yfJvyLj68coTRV
Recieved image no. 15 with id BsMZgHtCAMdxWX6tSCpG
Recieved image no. 16 with id UCPPh12x98xZjUHynuto
Recieved image no. 17 with id VtXdPlwgjqywirLu8mj9
Recieved image no. 18 with id Sqr8TtyCvlqjvtir95ks
Recieved image no. 19 with id WWbfMzxk0LsK5Ph6qZ9R
Reci

## Use the image metadata to generate dense captions with openrouter

In [8]:
import concurrent.futures as ft

PROMPT ="""
You are an assistant to a researcher investigating a large language model visual reasoning capabilities
In **as much detail as possible**, describe the image. Remember that this is for research and all images have been sources from public datasets. You must describe all images in detail. **Omit descriptions of watermark text or images**.
"""
with open("./img-metadata.json", "r", encoding="utf-8") as file:
    metadata = json.load(file)

def get_caption(client, prompt, url):
    response = client.chat.completions.create(
                model="qwen/qwen3.6-flash",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {
                                "type": "image_url",
                                "image_url": {"url": url}
                            }
                        ]
                    }
                ]
            )
            
    # Extract model's text response
    full_response = response.choices[0].message.content.strip()
    return full_response

def get_openai_image_url(url):
    try:
        # 1. Download the data to check the actual format
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        
        # 2. Inspect the data (not the extension)
        img = Image.open(BytesIO(resp.content))
        actual_format = img.format.upper() if img.format else ""

        # 3. If it's already a standard web format, just return the URL
        if actual_format in ['PNG', 'JPEG', 'WEBP']:
            return url

        # 4. Otherwise (TIFF, BMP, etc.), convert to PNG and encode to Base64
        buffer = BytesIO()
        if img.mode not in ("RGB", "RGBA"):
            img = img.convert("RGBA")
        
        img.save(buffer, format="PNG")
        b64_data = base64.b64encode(buffer.getvalue()).decode('utf-8')
        
        return f"data:image/png;base64,{b64_data}"

    except Exception as e:
        return f"Error processing image: {e}"

In [6]:
client = opi.OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY")
        )
images = metadata.get("image_data")
caption_data = {}
try:
    bar = tqdmnote(images)
    for i, image in enumerate(bar):
        print(images[image])
        bar.write(f"Processing image no. {i}")
        url = images[image]["url"]
        label = images[image]["label"]
        caption = get_caption(client, PROMPT, get_openai_image_url(url))
        caption_data[image] = {"url": url, "label": label, "caption": caption}
        bar.write("Success!")
except:
    print("STOPPED MIDWAY...")
          

with open(f"Captions/captions_{time.strftime("%d-%m-%Y_%H-%M-%S")}.json", "w", encoding="utf-8") as file:
    json.dump(caption_data, file, indent=4)

  0%|          | 0/200 [00:00<?, ?it/s]

{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/kqPGqRb9aAj3cRzRCySq/original.jpg', 'label': '1'}
Processing image no. 0
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/vyzu4H89BV5IpdJOOjJG/original.jpg', 'label': '1'}
Processing image no. 1
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/5QfGY7vy3T0aN9e0ilBe/original.jpg', 'label': '0'}
Processing image no. 2
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/Ac9Nb9CxhxdHAhMhRP6X/original.jpg', 'label': '0'}
Processing image no. 3
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/hsKtNmnjmSosOQzGdvxP/original.jpg', 'label': '1'}
Processing image no. 4
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/CSvuldZIg1GYEmfvZipy/original.jpg', 'label': '1'}
Processing image no. 5
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/n0BnmdMGLRlzu6rQDbfp/original.jpg', 'label': '

In [9]:
## Temporary code to fix our dataset captions(short of dangerous images)
client = opi.OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY")
        )
images = metadata.get("image_data")
with open("./our-dataset-captions.json", "r", encoding="utf-8") as file:
    prev_data = json.load(file)
try:
    bar = tqdmnote(images)
    for i, image in enumerate(bar):
        if image in prev_data: continue
        print(images[image])
        bar.write(f"Processing image no. {i}")
        url = images[image]["url"]
        label = images[image]["label"]
        caption = get_caption(client, PROMPT, get_openai_image_url(url))
        prev_data[image] = {"url": url, "label": label, "caption": caption}
        bar.write("Success!")
except:
    print("STOPPED MIDWAY...")
          

with open(f"Captions/captions_{time.strftime("%d-%m-%Y_%H-%M-%S")}.json", "w", encoding="utf-8") as file:
    json.dump(prev_data, file, indent=4)

  0%|          | 0/51 [00:00<?, ?it/s]

{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/jlAzrTyq8JBXMnSLzf1l/original.jpg', 'label': '01'}
Processing image no. 17
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/EFxNryFxTwhgbaIBDkt5/original.jpg', 'label': '01'}
Processing image no. 46
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/HoSop6MewtTjmHRccmy1/original.jpg', 'label': '01'}
Processing image no. 47
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/W6Jgl8BB04i9clwqJYNH/original.jpg', 'label': '01'}
Processing image no. 48
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/rNBL93QGQsyPIIvFwYoX/original.jpg', 'label': '01'}
Processing image no. 49
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/owtqQbcIO70ei2Rnmnfr/original.jpg', 'label': '01'}
Processing image no. 50
Success!
